In [0]:
#Carregando os arquivos da camada bronze
caminho_bronze = "/Volumes/workspace/default/dados_brutos/acidentes_brasil.csv"

df_bronze = spark.read.csv(caminho_bronze, 
                           header=True, 
                           sep=";", 
                           inferSchema=False, 
                           encoding="utf-8")

display(df_bronze.limit(10))




In [0]:
print(df_bronze.columns)

In [0]:
#Arrumando os cabeçalhos da tabela 
df_prata = df_bronze

for coluna_antiga in df_prata.columns:
    coluna_nova = coluna_antiga.replace(" ", "_").lower()
    df_prata = df_prata.withColumnRenamed(coluna_antiga, coluna_nova)
    print(f" {coluna_antiga} -> {coluna_nova}")
    

In [0]:
#Arrumando os dados da coluna data, string -> date 
from pyspark.sql.functions import col, to_date

df_prata = (
    df_prata.withColumnRenamed("data_inversa", "data")
        .withColumn("data", to_date(col("data"), "dd/MM/yyyy")) 
)


display(df_prata.limit(10))
df_prata.printSchema()



In [0]:
from pyspark.sql.functions import col, count, when

#Conta quantas colunas tem antes de limpar
total_linhas = df_prata.count()

df_prata = df_prata.dropna(how = 'all')
df_prata = df_prata.dropDuplicates()

for coluna in df_prata.columns:
  df_prata = df_prata.withColumn(coluna, when(col(coluna).try_cast("string") == "NULL","N/I").otherwise(col(coluna)))

print(f'O dataframe teve {df_prata.count() - total_linhas} linhas removidas')
display(df_prata.limit(10))

In [0]:
#Criando um novo Dataframe a partir da UF - SP
df_sp = df_prata.filter("uf = 'SP'")

display(df_sp)

In [0]:
df_sp.createOrReplaceTempView('tabela_acidentes_sp')
